# Sample-adequacy 

In [1]:
# ============================================================
# Sample-adequacy analysis for CB-SEM - Manuscript 1 (DIC -> RIC/ROC -> RCC -> SCR; DTG moderates)
# RMSEA-based power (MacCallum, Browne & Sugawara, 1996) + regression-based check
# Requires: scipy  (pip install scipy)
# ============================================================
from scipy.stats import ncx2, ncf, f as f_dist
from scipy.optimize import brentq

# ---------------- 1. MODEL SETTINGS (edit if your final model differs) ----------------
N = 413
items = {"DIC": 5, "RIC": 5, "ROC": 5, "RCC": 5, "DTG": 5, "SCR": 5}
exog = ["DIC", "DTG"]
endo = ["RIC", "ROC", "RCC", "SCR"]
paths = [("DIC", "RIC"), ("DIC", "ROC"), ("DIC", "RCC"),          # H1, H2, H3
         ("RIC", "RCC"), ("ROC", "RCC"),                          # H4, H5
         ("DTG", "RCC"),                                          # DTG main effect (needed for H9a, H9b)
         ("RCC", "SCR")]                                          # H6 (RCC -> SCR)
direct_to_scr = [("DIC", "SCR"), ("RIC", "SCR"), ("ROC", "SCR")]  # optional direct paths
interactions = {"RCC": 2}                                          # RIC x DTG and ROC x DTG enter the RCC equation
ALPHA = 0.05

# ---------------- 2. FUNCTIONS ----------------
def sem_df(paths, extra_params=0):
    """df and free parameters. Assumes marker-variable scaling, no cross-loadings, no correlated
    errors, all exogenous latents correlated. extra_params = additional freed parameters
    (e.g. 1 for a RIC-ROC disturbance covariance)."""
    for s, t in paths:
        assert s in items and t in items and t in endo, f"Check path {s}->{t}"
    k, p = len(items), sum(items.values())
    moments = p * (p + 1) // 2
    measurement = (p - k) + p                               # free loadings + error variances
    latent = len(exog) * (len(exog) + 1) // 2 + len(endo) + len(paths) + extra_params
    npar = measurement + latent
    return dict(indicators=p, npar=npar, df=moments - npar)

def rmsea_power(df, n, eps0=0.05, epsa=0.08, alpha=ALPHA):
    l0, la = (n - 1) * df * eps0**2, (n - 1) * df * epsa**2
    if epsa > eps0:                                         # test of close fit
        return 1 - ncx2.cdf(ncx2.ppf(1 - alpha, df, l0), df, la)
    return ncx2.cdf(ncx2.ppf(alpha, df, l0), df, la)        # test of not-close fit

def min_n(df, target=0.80, **kw):
    g = lambda n: rmsea_power(df, n, **kw) - target
    return 5 if g(5) >= 0 else brentq(g, 5, 200000)

def reg_min_n(u, f2=0.15, alpha=ALPHA, power=0.90):
    """Regression-based minimum N for u predictors (same test as G*Power's fixed-model R2 test)."""
    def pw(n):
        d2 = n - u - 1
        return 1 - ncf.cdf(f_dist.ppf(1 - alpha, u, d2), u, d2, f2 * n) - power
    return int(brentq(pw, u + 3, 2000)) + 1

def n_predictors(paths):
    """Largest number of predictors in one equation, counting interaction terms."""
    counts = {}
    for s, t in paths:
        counts[t] = counts.get(t, 0) + 1
    for t, k in interactions.items():
        counts[t] = counts.get(t, 0) + k
    return max(counts.values()), counts

# ---------------- 3. RUN ----------------
specs = [
    ("Hypothesised paths",                   paths,                    0),
    ("+ RIC-ROC disturbance covariance",     paths,                    1),
    ("+ direct paths to SCR",                paths + direct_to_scr,    0),
]
print(f"Manuscript 1  (N = {N})\n")
for label, p_, extra in specs:
    r = sem_df(p_, extra)
    u, counts = n_predictors(p_)
    print(f"{label:<34} indicators={r['indicators']}  npar={r['npar']}  df={r['df']}")
    print(f"   RMSEA power (close fit: .05 vs .08) = {rmsea_power(r['df'], N):.3f}   "
          f"(not-close fit: .05 vs .01) = {rmsea_power(r['df'], N, 0.05, 0.01):.3f}")
    print(f"   Minimum N for .80 power = {min_n(r['df'], 0.80, eps0=0.05, epsa=0.08):.0f}   "
          f"N : parameters = {N / r['npar']:.2f}")
    print(f"   Predictors per equation: {counts}")
    print(f"   Regression-based check: max predictors = {u}  ->  minimum N = {reg_min_n(u)}\n")

Manuscript 1  (N = 413)

Hypothesised paths                 indicators=30  npar=68  df=397
   RMSEA power (close fit: .05 vs .08) = 1.000   (not-close fit: .05 vs .01) = 1.000
   Minimum N for .80 power = 56   N : parameters = 6.07
   Predictors per equation: {'RIC': 1, 'ROC': 1, 'RCC': 6, 'SCR': 1}
   Regression-based check: max predictors = 6  ->  minimum N = 123

+ RIC-ROC disturbance covariance   indicators=30  npar=69  df=396
   RMSEA power (close fit: .05 vs .08) = 1.000   (not-close fit: .05 vs .01) = 1.000
   Minimum N for .80 power = 56   N : parameters = 5.99
   Predictors per equation: {'RIC': 1, 'ROC': 1, 'RCC': 6, 'SCR': 1}
   Regression-based check: max predictors = 6  ->  minimum N = 123

+ direct paths to SCR              indicators=30  npar=71  df=394
   RMSEA power (close fit: .05 vs .08) = 1.000   (not-close fit: .05 vs .01) = 1.000
   Minimum N for .80 power = 56   N : parameters = 5.82
   Predictors per equation: {'RIC': 1, 'ROC': 1, 'RCC': 6, 'SCR': 4}
   Regressi